# DNX Cauchos S.R.L. — 2. Solución de Big Data en Databricks

| N° | Punto de la rúbrica | Sección del notebook |
|---|---|---|
| 1 | Identificación de los datos | 1 |
| 2 | Arquitectura | 2 |
| 3 | Procesamiento con Databricks (DataFrames de Apache Spark) | 3 |
| 4 | Análisis exploratorio (EDA) | 4 |
| 5 | Limpieza y preparación | 5 |
| 6 | Persistencia (tablas de Databricks) | 6 |
| 7 | Indicadores y visualizaciones | 7 |
| 8 | Inteligencia artificial (Genie) | 8 |

## 1. Identificación de los datos
Se reutilizan los datos del proyecto de BI: el Data Mart `dm_DNX_Cauchos_SRL` exportado desde SQL Server a 6 archivos CSV. Las dimensiones cliente, producto y vendedor se exportaron multiplicadas x400 (`CROSS JOIN`) para trabajar con mayor volumen y con duplicados reales.

| Archivo | Contenido | Filas |
|---|---|---|
| `dim_cliente.csv` | Clientes: código, nombre, tipo, DNI/RUC, teléfono, distrito | 240 000 |
| `dim_producto.csv` | Suelas: nombre, talla, tipo de suela, stock, costo, precio, categoría, material | 24 000 |
| `dim_vendedor.csv` | Vendedores: DNI, nombre, apellido, usuario, cargo | 24 000 |
| `dim_metodoPago.csv` | Métodos de pago | 4 |
| `dim_tiempo.csv` | Calendario 2025-2026 | 688 |
| `Hecho_ventas.csv` | Líneas de venta: cantidad, precio, monto, descuento, total, costo, utilidad | 2 828 |

**Problemática del negocio:** controlar los descuentos frente al tope anual de S/ 30,000 y conocer qué productos, vendedores y distritos generan más ventas en 2025 y 2026.

## 2. Arquitectura
`SQL Server (Data Mart)` → `Exportación SSMS a CSV` → `Databricks Volume` → `Apache Spark (DataFrames en PySpark)` → `Tablas Delta de Databricks` → `Dashboard de Databricks` + `Genie`

(El gráfico de la arquitectura se incluye en el informe.)

## 3. Procesamiento con Databricks — carga en DataFrames de Apache Spark

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Ubicación de los CSV (Volume) y del esquema donde se guardarán las tablas
RUTA    = "/Volumes/prueba/prueba/ala"
CATALOG = "prueba"
SCHEMA  = "dnx_cauchos"

TOPE_DESCUENTO  = 30000.0   # tope anual de descuentos (S/)
UMBRAL_AMARILLO = 0.80      # alerta amarilla desde el 80 % del tope; roja al superarlo

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")

display(dbutils.fs.ls(RUTA))

In [ ]:
archivos = {f.name.lower(): f.path for f in dbutils.fs.ls(RUTA)}

def leer_csv(nombre):
    df = (spark.read.format("csv")
          .option("header", True)
          .option("inferSchema", False)      # se lee como texto para revisar el dato original
          .option("encoding", "UTF-8")
          .load(archivos[nombre]))
    # withColumnRenamed: quitar el carácter BOM que SSMS deja en la primera cabecera
    for c in df.columns:
        if c != c.replace("﻿", "").strip():
            df = df.withColumnRenamed(c, c.replace("﻿", "").strip())
    return df

df_cliente   = leer_csv("dim_cliente.csv")
df_producto  = leer_csv("dim_producto.csv")
df_vendedor  = leer_csv("dim_vendedor.csv")
df_metodo    = leer_csv("dim_metodopago.csv")
df_tiempo    = leer_csv("dim_tiempo.csv")
df_ventas    = leer_csv("hecho_ventas.csv")

dfs = {"dim_cliente": df_cliente, "dim_producto": df_producto, "dim_vendedor": df_vendedor,
       "dim_metodoPago": df_metodo, "dim_tiempo": df_tiempo, "hecho_ventas": df_ventas}

for nombre, df in dfs.items():
    print(f"{nombre:<15} {df.count():>9,} filas  {len(df.columns)} columnas")

In [ ]:
df_ventas.printSchema()
display(df_ventas)
display(df_cliente)

## 4. Análisis exploratorio (EDA)
### 4.1 Volumen y registros duplicados

In [ ]:
resumen = []
for nombre, df in dfs.items():
    total = df.count()
    distintos = df.distinct().count()
    resumen.append((nombre, total, distintos, total - distintos))

display(spark.createDataFrame(resumen, "tabla string, filas long, filas_distintas long, duplicados long"))

### 4.2 Estadística descriptiva (`describe`)

In [ ]:
MEDIDAS = ["cantidad", "precio_unitario", "monto_venta", "descuento", "total_venta", "costo_total", "utilidad"]
ventas_num = df_ventas.select([F.col(c).cast("double").alias(c) for c in MEDIDAS])
display(ventas_num.describe())

In [ ]:
display(df_producto.select(F.col("costo_fabricacion").cast("double"), F.col("precio").cast("double"),
                           F.col("stock").cast("int")).describe())

### 4.3 Valores nulos o vacíos por columna

In [ ]:
nulos = []
for nombre, df in dfs.items():
    fila = df.select([
        F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
        for c in df.columns]).first().asDict()
    nulos += [(nombre, c, int(v)) for c, v in fila.items() if v > 0]

display(spark.createDataFrame(nulos, "tabla string, columna string, nulos_o_vacios long"))

### 4.4 Distritos mal escritos (espacios y minúsculas)

In [ ]:
display(df_cliente
        .filter((F.col("distrito") != F.trim("distrito")) |
                (F.substring(F.trim("distrito"), 1, 1) == F.lower(F.substring(F.trim("distrito"), 1, 1))))
        .groupBy(F.concat(F.lit("["), F.col("distrito"), F.lit("]")).alias("distrito_original"))
        .count()
        .orderBy("distrito_original"))

### 4.5 Clientes con el mismo DNI/RUC y distinto código

In [ ]:
display(df_cliente.select("pk_cliente", "nombre_cliente", "dni").distinct()
        .groupBy("dni", "nombre_cliente")
        .agg(F.count("*").alias("codigos"), F.array_sort(F.collect_list("pk_cliente")).alias("lista_codigos"))
        .filter("codigos > 1")
        .orderBy("dni"))

### 4.6 Comportamiento de las ventas

In [ ]:
# Ventas y descuento por año (id_tiempo 1 = 2025-01-02, primer día del calendario)
ventas_tiempo = df_ventas.withColumn(
    "fecha", F.date_add(F.lit("2025-01-02").cast("date"), F.col("id_tiempo").cast("int") - 1))

display(ventas_tiempo.groupBy(F.year("fecha").alias("anio"))
        .agg(F.countDistinct("pk_venta").alias("ventas"),
             F.round(F.sum(F.col("total_venta").cast("double")), 2).alias("ventas_netas"),
             F.round(F.sum(F.col("descuento").cast("double")), 2).alias("descuento"),
             F.round(F.sum(F.col("utilidad").cast("double")), 2).alias("utilidad"))
        .orderBy("anio"))

In [ ]:
display(ventas_tiempo.groupBy(F.date_format("fecha", "yyyy-MM").alias("mes"))
        .agg(F.round(F.sum(F.col("total_venta").cast("double")), 2).alias("ventas_netas"))
        .orderBy("mes"))

### 4.7 Valores atípicos (método IQR) y ventas con pérdida

In [ ]:
cuartiles = ventas_num.agg(*[F.percentile_approx(c, [0.25, 0.75]).alias(c) for c in MEDIDAS]).first()
atipicos = []
for c in MEDIDAS:
    q1, q3 = cuartiles[c]
    li, ls = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    atipicos.append((c, float(q1), float(q3), float(li), float(ls),
                     ventas_num.filter((F.col(c) < li) | (F.col(c) > ls)).count()))
display(spark.createDataFrame(atipicos, "medida string, q1 double, q3 double, limite_inferior double, limite_superior double, atipicos long"))

display(ventas_num.filter(F.col("utilidad") < 0)
        .agg(F.count("*").alias("lineas_con_perdida"),
             F.round(F.sum("utilidad"), 2).alias("perdida_total")))

### 4.8 Relación entre la tabla de ventas y las dimensiones
Se comprueba si el `id_producto` de las ventas apunta al producto correcto: el costo unitario de la venta (`costo_total / cantidad`) debe ser igual al `costo_fabricacion` del producto.

In [ ]:
prod = (df_producto.select(F.col("id_producto").cast("int").alias("id_producto"), "pk_producto",
                           F.col("costo_fabricacion").cast("double").alias("costo_fab"))
        .distinct())
vta = df_ventas.select(F.col("id_producto").cast("int").alias("id_producto"),
                       (F.col("costo_total").cast("double") / F.col("cantidad").cast("double")).alias("costo_unit"))

def coincidencia(df):
    return df.agg(F.count("*").alias("lineas"),
                  F.sum(F.when(F.col("pk_producto").isNull(), 1).otherwise(0)).alias("sin_producto"),
                  F.round(100 * F.avg(F.when(F.abs(F.col("costo_unit") - F.col("costo_fab")) < 0.001, 1.0)
                                        .otherwise(0.0)), 2).alias("pct_costo_coincide"))

print("A) Uniendo por id_producto:")
display(coincidencia(vta.join(prod, "id_producto", "left")))

print("B) Uniendo por el número del código del producto (PRO-00003 -> 3):")
prod_codigo = prod.drop("id_producto").withColumn("id_producto", F.regexp_extract("pk_producto", r"(\d+)$", 1).cast("int"))
display(coincidencia(vta.join(prod_codigo, "id_producto", "left")))

# Rango de claves: las dimensiones empiezan en 2 y las ventas en 1
rangos = []
for nombre, df, col in [("dim_producto", df_producto, "id_producto"), ("dim_vendedor", df_vendedor, "id_vendedor"),
                        ("dim_metodoPago", df_metodo, "id_metodoP"), ("dim_tiempo", df_tiempo, "id_tiempo")]:
    d = df.agg(F.min(F.col(col).cast("int")), F.max(F.col(col).cast("int"))).first()
    v = df_ventas.agg(F.min(F.col(col).cast("int")), F.max(F.col(col).cast("int"))).first()
    rangos.append((nombre, d[0], d[1], v[0], v[1]))
display(spark.createDataFrame(rangos, "dimension string, min_dimension int, max_dimension int, min_ventas int, max_ventas int"))

**Resultado del EDA:**
* Las dimensiones cliente, producto y vendedor están repetidas 400 veces.
* `telefono` tiene valores vacíos y `distrito` tiene espacios y minúsculas.
* Hay clientes registrados dos veces con el mismo DNI/RUC.
* Las claves `id_*` de las dimensiones están desplazadas en +1 respecto a las ventas: unir por `id_producto` asigna el producto equivocado. Uniendo por el código del producto, el costo coincide al 100 %.

## 5. Limpieza y preparación
### 5.1 Clientes

In [ ]:
cliente = (df_cliente
    .select("pk_cliente", "nombre_cliente", "tipo_cliente", "dni", "telefono", "distrito")   # select
    .dropDuplicates()                                                                        # quita las 400 repeticiones
    .withColumn("pk_cliente", F.upper(F.trim("pk_cliente")))
    .withColumn("nombre_cliente", F.trim("nombre_cliente"))
    .withColumn("dni", F.trim("dni"))
    .withColumn("telefono", F.when(F.trim("telefono") == "", F.lit(None)).otherwise(F.trim("telefono")))
    .withColumn("distrito", F.regexp_replace(F.trim("distrito"), r"\s+", " ")))

# Distrito: se toma la escritura más frecuente de cada distrito (groupBy) -> 'ate' pasa a 'Ate'
w = Window.partitionBy("clave").orderBy(F.desc("n"))
distrito_ok = (cliente.withColumn("clave", F.lower("distrito"))
               .groupBy("clave", "distrito").agg(F.count("*").alias("n"))
               .withColumn("r", F.row_number().over(w)).filter("r = 1")
               .select("clave", F.col("distrito").alias("distrito_ok")))

cliente = (cliente.withColumn("clave", F.lower("distrito"))
           .join(distrito_ok, "clave")
           .drop("clave", "distrito")
           .withColumnRenamed("distrito_ok", "distrito")                                    # withColumnRenamed
           .withColumn("id_cliente", F.regexp_extract("pk_cliente", r"(\d+)$", 1).cast("int")))

# Clientes duplicados por DNI: se conserva el código más antiguo
w_dni = Window.partitionBy("dni").orderBy("id_cliente")
cliente = cliente.withColumn("id_cliente_final", F.first("id_cliente").over(w_dni))
duplicados = cliente.filter("id_cliente <> id_cliente_final").select("id_cliente", "id_cliente_final")
display(duplicados)

cliente = (cliente.filter("id_cliente = id_cliente_final")                                    # filter
           .drop("id_cliente_final")
           .na.fill({"telefono": "SIN REGISTRO"})                                               # na
           .select("id_cliente", "pk_cliente", "nombre_cliente", "tipo_cliente", "dni", "telefono", "distrito"))

print("Clientes:", df_cliente.count(), "->", cliente.count())
display(cliente.groupBy("distrito").count().orderBy("distrito"))

### 5.2 Productos, vendedores, métodos de pago y tiempo
La clave `id_*` se reconstruye a partir del código de negocio (`PRO-00003` → 3) para que coincida con la tabla de ventas.

In [ ]:
producto = (df_producto
    .select("pk_producto", "nombre_producto", "tipo_suela", "talla", "stock", "costo_fabricacion", "precio", "categoria", "material")
    .dropDuplicates()
    .withColumn("id_producto", F.regexp_extract("pk_producto", r"(\d+)$", 1).cast("int"))
    .withColumn("talla", F.col("talla").cast("int"))
    .withColumn("stock", F.col("stock").cast("int"))
    .withColumn("costo_fabricacion", F.col("costo_fabricacion").cast("decimal(10,2)"))
    .withColumn("precio", F.col("precio").cast("decimal(10,2)"))
    .filter((F.col("precio") > 0) & (F.col("costo_fabricacion") > 0)))

vendedor = (df_vendedor
    .select("pk_vendedor", "dni", "nombre_vendedor", "apellido", "usuario", "cargo")
    .dropDuplicates()
    .withColumn("id_vendedor", F.regexp_extract("pk_vendedor", r"(\d+)$", 1).cast("int"))
    .withColumn("vendedor", F.concat_ws(" ", F.trim("nombre_vendedor"), F.trim("apellido"))))

metodo_pago = (df_metodo
    .select("pk_metodoP", "nombre_metodo", "tipo_pago")
    .dropDuplicates()
    .withColumnRenamed("pk_metodoP", "pk_metodo_pago")
    .withColumn("id_metodo_pago", F.regexp_extract("pk_metodo_pago", r"(\d+)$", 1).cast("int")))

tiempo = (df_tiempo
    .select("fecha", "nombre_mes")
    .dropDuplicates()
    .withColumn("fecha", F.to_date("fecha"))
    .withColumn("id_tiempo", F.datediff("fecha", F.lit("2025-01-02").cast("date")) + 1)   # 1 = primer día
    .withColumn("dia", F.dayofmonth("fecha"))
    .withColumn("mes", F.month("fecha"))
    .withColumn("trimestre", F.quarter("fecha"))
    .withColumn("anio", F.year("fecha")))

for n, df in [("producto", producto), ("vendedor", vendedor), ("metodo_pago", metodo_pago), ("tiempo", tiempo)]:
    print(f"{n:<12} {df.count():>5} filas")

### 5.3 Ventas

In [ ]:
ventas = (df_ventas
    .select("pk_venta", "pk_detalle", "id_cliente", "id_producto", "id_vendedor", "id_metodoP", "id_tiempo", *MEDIDAS)
    .withColumnRenamed("id_metodoP", "id_metodo_pago")
    .dropDuplicates(["pk_detalle"]))

for c in ["id_cliente", "id_producto", "id_vendedor", "id_metodo_pago", "id_tiempo", "cantidad"]:
    ventas = ventas.withColumn(c, F.col(c).cast("int"))
for c in ["precio_unitario", "monto_venta", "descuento", "total_venta", "costo_total", "utilidad"]:
    ventas = ventas.withColumn(c, F.col(c).cast("decimal(12,2)"))

# Ventas de clientes duplicados -> cliente que se conserva
ventas = (ventas.join(duplicados, "id_cliente", "left")
                .withColumn("id_cliente", F.coalesce("id_cliente_final", "id_cliente"))
                .drop("id_cliente_final"))

# Reglas de calidad
antes = ventas.count()
ventas = (ventas.na.drop(subset=["pk_venta", "pk_detalle", "cantidad", "total_venta"])
                .filter((F.col("cantidad") > 0) & (F.col("precio_unitario") > 0))
                .filter((F.col("descuento") >= 0) & (F.col("descuento") <= F.col("monto_venta")))
                .filter(F.abs(F.col("total_venta") - (F.col("monto_venta") - F.col("descuento"))) <= 0.01))
# Evidencia de los cambios
print("Líneas de clientes duplicados reasignadas:",
      df_ventas.filter(F.col("id_cliente").cast("int").between(596, 600)).count())
print("Líneas que aún apuntan a CLI-00596 a 600:",
      ventas.filter(F.col("id_cliente").between(596, 600)).count())


In [ ]:
# Comprobación: todas las ventas encuentran su cliente, producto, vendedor, método y fecha
for nombre, df, k in [("cliente", cliente, "id_cliente"), ("producto", producto, "id_producto"),
                      ("vendedor", vendedor, "id_vendedor"), ("metodo_pago", metodo_pago, "id_metodo_pago"),
                      ("tiempo", tiempo, "id_tiempo")]:
    print(f"{nombre:<12} ventas sin relación: {ventas.join(df.select(k), k, 'left_anti').count()}")

In [ ]:
# Resumen antes / después de la limpieza
display(spark.createDataFrame([
    ("cliente",     df_cliente.count(),  cliente.count()),
    ("producto",    df_producto.count(), producto.count()),
    ("vendedor",    df_vendedor.count(), vendedor.count()),
    ("metodo_pago", df_metodo.count(),   metodo_pago.count()),
    ("tiempo",      df_tiempo.count(),   tiempo.count()),
    ("ventas",      df_ventas.count(),   ventas.count()),
], "tabla string, filas_antes long, filas_despues long"))

## 6. Persistencia — tablas de Databricks

In [ ]:
def guardar(df, tabla):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.{tabla}")
    print(f"✔ {tabla}")

guardar(cliente, "cliente")
guardar(producto, "producto")
guardar(vendedor, "vendedor")
guardar(metodo_pago, "metodo_pago")
guardar(tiempo, "tiempo")
guardar(ventas, "ventas")

# Tabla de ventas con los datos de cada dimensión (para el dashboard y Genie)
ventas_detalle = (spark.table("ventas")
    .join(spark.table("tiempo").select("id_tiempo", "fecha", "anio", "mes", "nombre_mes"), "id_tiempo")
    .join(spark.table("cliente").select("id_cliente", "nombre_cliente", "tipo_cliente", "distrito"), "id_cliente")
    .join(spark.table("producto").select("id_producto", "nombre_producto", "talla", "categoria", "material"), "id_producto")
    .join(spark.table("vendedor").select("id_vendedor", "vendedor", "cargo"), "id_vendedor")
    .join(spark.table("metodo_pago").select("id_metodo_pago", "nombre_metodo"), "id_metodo_pago")
    .select("pk_venta", "pk_detalle", "fecha", "anio", "mes", "nombre_mes", "nombre_cliente", "tipo_cliente",
            "distrito", "nombre_producto", "talla", "categoria", "material", "vendedor", "cargo", "nombre_metodo",
            "cantidad", "precio_unitario", "monto_venta", "descuento", "total_venta", "costo_total", "utilidad"))
guardar(ventas_detalle, "ventas_detalle")

In [ ]:
%sql
SHOW TABLES IN prueba.dnx_cauchos

## 7. Indicadores y visualizaciones
En cada resultado se puede pulsar **+ → Visualization** para verlo como gráfico. Las tablas de indicadores se guardan para usarlas en el dashboard de Databricks.
### 7.1 Indicadores generales por año

In [ ]:
kpi_anual = (spark.table("ventas_detalle").groupBy("anio")
    .agg(F.countDistinct("pk_venta").alias("ventas"),
         F.sum("cantidad").alias("unidades"),
         F.sum("total_venta").alias("ventas_netas"),
         F.sum("descuento").alias("descuento"),
         F.sum("costo_total").alias("costo"),
         F.sum("utilidad").alias("utilidad"))
    .withColumn("margen_pct", F.round(100 * F.col("utilidad") / F.col("ventas_netas"), 2))
    .orderBy("anio"))
guardar(kpi_anual, "kpi_anual")
display(spark.table("kpi_anual"))

### 7.2 Control de descuentos frente al tope de S/ 30,000
Alerta **VERDE** < 80 % del tope · **AMARILLA** 80 %–100 % · **ROJA** > 100 %.

In [ ]:
def alerta(pct):
    return F.when(pct > 1, "ROJA").when(pct >= UMBRAL_AMARILLO, "AMARILLA").otherwise("VERDE")

kpi_descuento_anual = (spark.table("ventas_detalle").groupBy("anio")
    .agg(F.sum("descuento").cast("double").alias("descuento_total"))
    .withColumn("tope", F.lit(TOPE_DESCUENTO))
    .withColumn("saldo", F.col("tope") - F.col("descuento_total"))
    .withColumn("pct_consumo", F.round(100 * F.col("descuento_total") / F.col("tope"), 1))
    .withColumn("alerta", alerta(F.col("descuento_total") / F.col("tope")))
    .orderBy("anio"))
guardar(kpi_descuento_anual, "kpi_descuento_anual")
display(spark.table("kpi_descuento_anual"))

In [ ]:
w_acum = Window.partitionBy("anio").orderBy("mes").rowsBetween(Window.unboundedPreceding, Window.currentRow)
kpi_descuento_mensual = (spark.table("ventas_detalle").groupBy("anio", "mes", "nombre_mes")
    .agg(F.sum("descuento").cast("double").alias("descuento_mes"))
    .withColumn("descuento_acumulado", F.sum("descuento_mes").over(w_acum))
    .withColumn("tope", F.lit(TOPE_DESCUENTO))
    .withColumn("alerta", alerta(F.col("descuento_acumulado") / F.col("tope")))
    .orderBy("anio", "mes"))
guardar(kpi_descuento_mensual, "kpi_descuento_mensual")
display(spark.table("kpi_descuento_mensual"))

### 7.3 Productos, vendedores y distritos con más ventas

In [ ]:
w_rank = Window.partitionBy("anio").orderBy(F.desc("ventas_netas"))

def ranking(columnas, tabla):
    df = (spark.table("ventas_detalle").groupBy("anio", *columnas)
          .agg(F.sum("cantidad").alias("unidades"),
               F.sum("total_venta").alias("ventas_netas"),
               F.sum("descuento").alias("descuento"),
               F.sum("utilidad").alias("utilidad"))
          .withColumn("puesto", F.dense_rank().over(w_rank))
          .orderBy("anio", "puesto"))
    guardar(df, tabla)
    return spark.table(tabla)

kpi_productos  = ranking(["nombre_producto", "categoria"], "kpi_productos")
kpi_vendedores = ranking(["vendedor", "cargo"], "kpi_vendedores")
kpi_distritos  = ranking(["distrito"], "kpi_distritos")

display(kpi_productos.filter("puesto <= 10"))
display(kpi_vendedores.filter("puesto <= 10"))
display(kpi_distritos.filter("puesto <= 10"))

### 7.4 Dashboard
Crear en **Dashboards → Create dashboard** usando las tablas `kpi_anual`, `kpi_descuento_anual`, `kpi_descuento_mensual`, `kpi_productos`, `kpi_vendedores` y `kpi_distritos`, con un filtro por `anio` (2025 / 2026).

In [ ]:
display(spark.table("kpi_vendedores"))

In [ ]:
display(spark.table("kpi_productos"))

In [ ]:
display(spark.table("kpi_distritos"))

In [ ]:
display(spark.table("kpi_descuento_mensual"))

In [ ]:
display(spark.table("kpi_descuento_anual"))

In [ ]:
display(spark.table("kpi_anual"))

## 8. Inteligencia artificial — Databricks Genie
Crear un espacio en **Genie → New** con las tablas `ventas_detalle`, `kpi_anual`, `kpi_descuento_anual`, `kpi_descuento_mensual`, `kpi_productos`, `kpi_vendedores` y `kpi_distritos`, y hacer preguntas en lenguaje natural, por ejemplo:
* ¿Cuánto descuento se otorgó en 2026 y qué porcentaje del tope de S/ 30,000 representa?
* ¿En qué mes de 2026 el descuento acumulado superó el tope?
* ¿Cuáles son los 5 productos con más ventas netas en 2025?
* ¿Qué vendedor tuvo más ventas en 2026?
* ¿Qué distritos tienen más ventas?

La declaración de uso de IA se incluye en el informe.